<a href="https://colab.research.google.com/github/alpacaYiChun/ML/blob/master/Alimama_TwinTower.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FULL RUNNABLE COLAB CELL — UPDATED EXACTLY PER YOUR REQUEST
#
# CHANGES ONLY:
# 1) REAL explicit L2 regularization added (true penalty term in loss),
#    configurable separately for item tower / user tower / embeddings / bias.
#    -> Optimizer changed from AdamW to Adam(weight_decay=0.0)
#    -> loss = lossA + 3*lossB + explicit_l2
#
# 2) History encoder is configurable:
#    -> HISTORY_ENCODER = "gru" or "transformer"
#    -> same interface, same rest of pipeline
#
# 3) Recall now also reports:
#    -> @100, @200, @500, @1000
#
# EVERYTHING ELSE kept unchanged.
# ============================================================

!pip -q install tqdm==4.66.4

import os, tarfile, random, math, copy
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# -----------------------
# Paths + download
# -----------------------
DATA_DIR = "/content/taobao_ad"
os.makedirs(DATA_DIR, exist_ok=True)

files = {
    "ad_feature.csv.tar":   "https://zenodo.org/records/8088629/files/ad_feature.csv.tar?download=1",
    "raw_sample.csv.tar":   "https://zenodo.org/records/8088629/files/raw_sample.csv.tar?download=1",
    "user_profile.csv.tar": "https://zenodo.org/records/8088629/files/user_profile.csv.tar?download=1",
}

def extract_tar(path):
    with tarfile.open(path, "r") as tar:
        tar.extractall(os.path.dirname(path))

for fname, url in files.items():
    out_path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(out_path):
        print("Downloading", fname)
        !wget -q --show-progress -O "$out_path" "$url"
    extract_tar(out_path)

AD_PATH  = os.path.join(DATA_DIR, "ad_feature.csv")
RAW_PATH = os.path.join(DATA_DIR, "raw_sample.csv")
UP_PATH  = os.path.join(DATA_DIR, "user_profile.csv")
for p in [AD_PATH, RAW_PATH, UP_PATH]:
    if not os.path.exists(p): raise FileNotFoundError(p)
print("Files ready")

# -----------------------
# Config (industry-ish defaults; adjust if OOM)
# -----------------------
MAX_RAW_ROWS = None  # debug: set e.g. 5_000_000

# TopK by coverage (NOT hard-coded)
TARGET_CLICK_COVERAGE_USER = 0.95   # cover 95% of CLICK volume by top users
TARGET_CLICK_COVERAGE_ITEM = 0.95   # cover 95% of CLICK volume by top items
TOPK_CAP_USER = 200_000             # safety caps
TOPK_CAP_ITEM = 200_000

# Active/hot pool: tune thresholds to reach coverage on labels
RECENT_DAYS = 14                    # "recent" relative to dataset end_ts
TARGET_POOL_COVERAGE_VAL  = 0.95
TARGET_POOL_COVERAGE_TEST = 0.95

# Sequences / loader
MAX_HIST_CLICKS = 50
BATCH_SIZE = 1024
NUM_WORKERS = 2

# Embeddings / model (item tower unchanged; history encoder configurable)
EMB_DIM = 64
GRU_HIDDEN = 256
MLP_HIDDEN = 512
DROPOUT = 0.1

# History encoder config
HISTORY_ENCODER = "transformer"   # "gru" or "transformer"

# Transformer history config
TRANSFORMER_NHEAD = 8
TRANSFORMER_FF_DIM = 1024
TRANSFORMER_LAYERS = 2
TRANSFORMER_DROPOUT = 0.1

# Loss setup
K_NEG = 255
LOSS_B_MULT = 3.0

# Optim
LR = 2e-4
GRAD_CLIP = 1.0

# REAL explicit L2 regularization (penalty added to loss)
# Keep configurable so it is truly explicit and not hidden in optimizer weight_decay.
USE_EXPLICIT_L2 = True

# Separate coefficients so regularization is truly configurable "for every model"
L2_ITEM = 1e-8
L2_USER = 1e-8

# What to include in explicit L2
L2_INCLUDE_EMBEDDINGS = True
L2_INCLUDE_BIAS = True

# Training
EPOCHS = 20
TAU_START = 0.20
TAU_END   = 0.07
TAU_WARMUP_EPOCHS = 5
TAU_VAL = 0.07

# -----------------------
# Load data
# -----------------------
ad = pd.read_csv(AD_PATH, low_memory=False)
up = pd.read_csv(UP_PATH, low_memory=False)
raw = pd.read_csv(RAW_PATH, nrows=MAX_RAW_ROWS, header=None, low_memory=False, dtype="string")
raw.columns = raw.iloc[0]
raw = raw.iloc[1:].reset_index(drop=True)

# -----------------------
# Clean + types
# -----------------------
ad["adgroup_id"] = pd.to_numeric(ad["adgroup_id"], errors="coerce").astype("Int64")
ad["cate_id"]    = pd.to_numeric(ad["cate_id"], errors="coerce").astype("Int64")
ad["brand"]      = pd.to_numeric(ad["brand"], errors="coerce").astype("Int64")
ad["price"]      = pd.to_numeric(ad["price"], errors="coerce")

up = up.drop(columns=["pvalue_level","new_user_class_level"], errors="ignore")
up["userid"] = pd.to_numeric(up["userid"], errors="coerce").astype("Int64")
for c in ["cms_segid","cms_group_id","final_gender_code","age_level","shopping_level","occupation"]:
    up[c] = pd.to_numeric(up[c], errors="coerce").fillna(0).astype(np.int64)

raw["user"]       = pd.to_numeric(raw["user"], errors="coerce").astype("Int64")
raw["adgroup_id"] = pd.to_numeric(raw["adgroup_id"], errors="coerce").astype("Int64")
raw["clk"]        = pd.to_numeric(raw["clk"], errors="coerce").fillna(0).astype(np.int64)
raw["nonclk"]     = pd.to_numeric(raw["nonclk"], errors="coerce").fillna(0).astype(np.int64)
raw["time_stamp"] = pd.to_numeric(raw["time_stamp"], errors="coerce").fillna(0).astype(np.int64)
raw["pid"]        = raw["pid"].astype("string")

dt = pd.to_datetime(raw["time_stamp"], unit="s", utc=True, errors="coerce")
raw["year"]  = (dt.dt.year.fillna(2000).astype(np.int64) - 2000).clip(lower=0)
raw["month"] = dt.dt.month.fillna(1).astype(np.int64)

raw = raw[raw["user"].notna() & raw["adgroup_id"].notna()].reset_index(drop=True)
print("raw rows:", len(raw))

# -----------------------
# Helper: choose TopK by click coverage
# -----------------------
def topk_by_click_coverage(ids: np.ndarray, is_click: np.ndarray, target_cov: float, cap: int):
    m = (is_click.astype(np.int64) == 1)
    ids_click = ids[m]
    vc = pd.Series(ids_click).value_counts().sort_values(ascending=False)
    counts = vc.to_numpy(dtype=np.int64)
    total = counts.sum()
    if total == 0:
        return {}, 0, 0.0
    cumsum = np.cumsum(counts)
    k = int(np.searchsorted(cumsum, target_cov * total) + 1)
    k = min(k, cap, len(vc))
    keep = vc.head(k).index.to_numpy()
    id2idx = {int(kid): i+1 for i, kid in enumerate(keep)}  # 0=OOV
    achieved = float(cumsum[k-1] / total) if k > 0 else 0.0
    return id2idx, k, achieved

u_raw_vals = raw["user"].astype(np.int64).values
i_raw_vals = raw["adgroup_id"].astype(np.int64).values
clk_vals   = raw["clk"].astype(np.int64).values

user_id2idx, USER_TOPK, user_cov = topk_by_click_coverage(
    u_raw_vals, clk_vals, TARGET_CLICK_COVERAGE_USER, TOPK_CAP_USER
)
adg_id2idx, ADG_TOPK, item_cov = topk_by_click_coverage(
    i_raw_vals, clk_vals, TARGET_CLICK_COVERAGE_ITEM, TOPK_CAP_ITEM
)

print(f"USER_TOPK chosen={USER_TOPK} covers click volume={user_cov:.4f} (target={TARGET_CLICK_COVERAGE_USER})")
print(f"ADG_TOPK  chosen={ADG_TOPK} covers click volume={item_cov:.4f} (target={TARGET_CLICK_COVERAGE_ITEM})")

# cate full
cate_unique = pd.Index(ad["cate_id"].dropna().astype(np.int64).unique())
cate_id2idx = {int(k): i+1 for i, k in enumerate(cate_unique)}  # 0=NA/OOV

# brand topk by coverage on ad table occurrences (fallback)
def topk_id2idx(values: np.ndarray, topk: int):
    vc = pd.Series(values).value_counts()
    keep = vc.head(topk).index.to_numpy()
    return {int(k): i+1 for i, k in enumerate(keep)}  # 0=OOV

BRAND_TOPK = 2000
brand_vals = ad["brand"].dropna().astype(np.int64).values
brand_id2idx = topk_id2idx(brand_vals, BRAND_TOPK)

pid_unique = pd.Index(raw["pid"].dropna().unique())
pid2idx = {str(k): i+1 for i, k in enumerate(pid_unique)}  # 0 padding

USER_V  = len(user_id2idx) + 1
ADG_V   = len(adg_id2idx) + 1
CATE_V  = len(cate_id2idx) + 1
BRAND_V = len(brand_id2idx) + 1
YEAR_V  = int(raw["year"].max()) + 2
MONTH_V = 13

CMS_SEG_V = int(up["cms_segid"].max()) + 2
CMS_GRP_V = int(up["cms_group_id"].max()) + 2
GENDER_V  = int(up["final_gender_code"].max()) + 2
AGE_V     = int(up["age_level"].max()) + 2
SHOP_V    = int(up["shopping_level"].max()) + 2
OCC_V     = int(up["occupation"].max()) + 2

print("Vocabs:", {
    "USER_V":USER_V, "ADG_V":ADG_V, "CATE_V":CATE_V, "BRAND_V":BRAND_V, "YEAR_V":YEAR_V
})

# -----------------------
# Item feature tables indexed by adg_code (TopK only)
# -----------------------
price = ad["price"].to_numpy(dtype=np.float32)
price = np.clip(np.nan_to_num(price, nan=0.0), 0, None)
lp = np.log1p(price)
mn, mx = float(lp.min()), float(lp.max())
price_norm_all = ((lp - mn) / (mx - mn + 1e-12)).astype(np.float32)

cate_by_adg  = np.zeros(ADG_V, dtype=np.int64)
brand_by_adg = np.zeros(ADG_V, dtype=np.int64)
price_by_adg = np.zeros(ADG_V, dtype=np.float32)

ad_id = ad["adgroup_id"].astype("Int64").to_numpy()
cate_id = ad["cate_id"].astype("Int64").to_numpy()
brand_id = ad["brand"].astype("Int64").to_numpy()

for idx in range(len(ad)):
    aid = ad_id[idx]
    if pd.isna(aid):
        continue
    adg_code = adg_id2idx.get(int(aid), 0)
    if adg_code == 0:
        continue
    cid = cate_id[idx]
    bid = brand_id[idx]
    ccode = cate_id2idx.get(int(cid), 0) if not pd.isna(cid) else 0
    bcode = brand_id2idx.get(int(bid), 0) if not pd.isna(bid) else 0
    cate_by_adg[adg_code]  = ccode
    brand_by_adg[adg_code] = bcode
    price_by_adg[adg_code] = float(price_norm_all[idx])

# -----------------------
# Build per-user sorted sequences (RAW ids in sequences; map only in __getitem__)
# -----------------------
raw_pid_code  = np.array([pid2idx.get(str(x), 0) for x in raw["pid"].values], dtype=np.int64)

tmp = pd.DataFrame({
    "user": raw["user"].astype(np.int64).values,
    "adg_raw": raw["adgroup_id"].astype(np.int64).values,
    "pid_code": raw_pid_code,
    "year": raw["year"].astype(np.int64).values,
    "month": raw["month"].astype(np.int64).values,
    "clk": raw["clk"].astype(np.int64).values,
    "non": raw["nonclk"].astype(np.int64).values,
    "ts": raw["time_stamp"].astype(np.int64).values
}).sort_values(["user","ts"]).reset_index(drop=True)

u_static = up.set_index(up["userid"].astype(np.int64))[[
    "cms_segid","cms_group_id","final_gender_code","age_level","shopping_level","occupation"
]]

user_event = {}
for u, g in tqdm(tmp.groupby("user", sort=False), desc="building user_event"):
    adg_raw = g["adg_raw"].to_numpy(np.int64)
    yr  = g["year"].to_numpy(np.int64)
    mo  = g["month"].to_numpy(np.int64)
    clk = g["clk"].to_numpy(np.int64)
    non = g["non"].to_numpy(np.int64)
    ts  = g["ts"].to_numpy(np.int64)

    click_pos = np.where(clk == 1)[0]
    if len(click_pos) < 3:
        continue

    try:
        us = u_static.loc[int(u)]
        demo = np.array([int(us[c]) for c in [
            "cms_segid","cms_group_id","final_gender_code",
            "age_level","shopping_level","occupation"
        ]], dtype=np.int64)
    except KeyError:
        demo = np.zeros((6,), dtype=np.int64)

    user_event[int(u)] = {
        "demo": demo,
        "adg_raw": adg_raw, "year": yr, "month": mo,
        "clk": clk, "non": non, "ts": ts,
        "click_pos": click_pos
    }

print("Users kept:", len(user_event))

# -----------------------
# Active/hot pool selection — AUTO TUNE for coverage
# recent window relative to dataset end_ts (NOT current time)
# -----------------------
end_ts = int(raw["time_stamp"].max())
cut_ts = end_ts - int(RECENT_DAYS * 86400)

raw_adg_codes = np.array([adg_id2idx.get(int(x), 0) for x in raw["adgroup_id"].astype(np.int64).values], dtype=np.int64)
raw_clk = raw["clk"].astype(np.int64).values
raw_ts  = raw["time_stamp"].astype(np.int64).values

click_mask = (raw_clk == 1) & (raw_adg_codes != 0)
recent_mask = click_mask & (raw_ts >= cut_ts)

total_clicks = np.bincount(raw_adg_codes[click_mask], minlength=ADG_V).astype(np.int64)
recent_clicks = np.bincount(raw_adg_codes[recent_mask], minlength=ADG_V).astype(np.int64)

def split_label_coverage(user_event_dict, split, active_mask):
    total, covered = 0, 0
    for u, d in user_event_dict.items():
        cp = d["click_pos"]
        if split == "val":
            use = cp[-2:-1]
        elif split == "test":
            use = cp[-1:]
        else:
            continue
        for p in use:
            total += 1
            pos_raw = int(d["adg_raw"][int(p)])
            pos_code = adg_id2idx.get(pos_raw, 0)
            if pos_code != 0 and active_mask[pos_code]:
                covered += 1
    return covered / max(total, 1), covered, total

min_recent = 50
min_total  = 1

best_active_mask = None
best_stats = None

for trial in range(12):
    active_mask = (recent_clicks >= min_recent) & (total_clicks >= min_total)
    active_mask[0] = False

    val_cov, val_cov_n, val_tot = split_label_coverage(user_event, "val", active_mask)
    test_cov, test_cov_n, test_tot = split_label_coverage(user_event, "test", active_mask)
    pool_size = int(active_mask.sum())

    print(f"[POOL] min_recent={min_recent:>3} pool={pool_size:>6}  "
          f"val_cov={val_cov:.3f} ({val_cov_n}/{val_tot})  test_cov={test_cov:.3f} ({test_cov_n}/{test_tot})")

    best_active_mask = active_mask
    best_stats = (min_recent, pool_size, val_cov, test_cov)

    if val_cov >= TARGET_POOL_COVERAGE_VAL and test_cov >= TARGET_POOL_COVERAGE_TEST:
        break

    min_recent = max(1, int(min_recent * 0.6))

active_mask = best_active_mask
ACTIVE_ADG_CODES = np.where(active_mask)[0].astype(np.int64)
print(f"Final active pool: size={len(ACTIVE_ADG_CODES)}, min_recent={best_stats[0]}, "
      f"val_cov={best_stats[2]:.3f}, test_cov={best_stats[3]:.3f}, end_ts={end_ts}, cut_ts={cut_ts}")

# negative sampling distribution within active pool: pop^0.75
pop = np.maximum(total_clicks.astype(np.float64), 1.0)
prob = np.power(pop, 0.75)
prob[~active_mask] = 0.0
prob[0] = 0.0
prob = prob / (prob.sum() + 1e-12)
prob_t = torch.from_numpy(prob.astype(np.float32))

# -----------------------
# Dataset (keep samples only if POS in active pool)
# - sequences built by RAW ids; mapping only in __getitem__
# -----------------------
class ClickDatasetActive(Dataset):
    def __init__(self, user_event_dict, split: str, K: int, max_hist=50):
        self.user_event = user_event_dict
        self.max_hist = max_hist
        self.K = int(K)
        self.samples = []

        for u, d in self.user_event.items():
            cp = d["click_pos"]
            if split == "train":
                use = cp[:-2]
            elif split == "val":
                use = cp[-2:-1]
            elif split == "test":
                use = cp[-1:]
            else:
                raise ValueError(split)

            for p in use:
                pos_raw = int(d["adg_raw"][int(p)])
                pos_code = adg_id2idx.get(pos_raw, 0)
                if pos_code == 0 or (not active_mask[pos_code]):
                    continue
                self.samples.append((u, int(p)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        u_raw, click_pos = self.samples[i]
        d = self.user_event[u_raw]
        cp = d["click_pos"]
        idx = np.searchsorted(cp, click_pos)

        hist_click_pos = cp[:idx]
        if len(hist_click_pos) > self.max_hist:
            hist_click_pos = hist_click_pos[-self.max_hist:]

        uc = user_id2idx.get(int(u_raw), 0)
        u_feats = np.concatenate([np.array([uc], dtype=np.int64), d["demo"]], axis=0)

        pos_raw = int(d["adg_raw"][click_pos])
        pos_adg = adg_id2idx.get(pos_raw, 0)

        if len(hist_click_pos) == 0:
            hist_adg  = np.array([0], dtype=np.int64)
            hist_year = np.array([0], dtype=np.int64)
            hist_mon  = np.array([0], dtype=np.int64)
        else:
            hist_adg_raw = d["adg_raw"][hist_click_pos].astype(np.int64)
            hist_adg = np.array([adg_id2idx.get(int(x), 0) for x in hist_adg_raw], dtype=np.int64)
            hist_year = d["year"][hist_click_pos].astype(np.int64)
            hist_mon  = d["month"][hist_click_pos].astype(np.int64)

        all_clicked_codes = np.array([adg_id2idx.get(int(x), 0) for x in d["adg_raw"][cp].tolist()], dtype=np.int64)
        all_clicked_codes = np.unique(all_clicked_codes)
        all_clicked_codes = all_clicked_codes[all_clicked_codes != 0]
        forbid = set(all_clicked_codes.tolist())
        forbid.add(int(pos_adg))

        negs = []
        need = self.K
        draw = int(self.K * 10)
        cand = torch.multinomial(prob_t, num_samples=draw, replacement=True).numpy().astype(np.int64)
        for c in cand:
            if c == 0:
                continue
            if not active_mask[c]:
                continue
            if c in forbid:
                continue
            negs.append(int(c))
            need -= 1
            if need == 0:
                break
        if need > 0:
            negs += [0] * need
        neg_adgs = np.asarray(negs, dtype=np.int64)

        pos_cate  = int(cate_by_adg[pos_adg]) if pos_adg else 0
        pos_brand = int(brand_by_adg[pos_adg]) if pos_adg else 0
        pos_price = float(price_by_adg[pos_adg]) if pos_adg else 0.0
        pos_item_cats = np.array([pos_adg, pos_cate, pos_brand], dtype=np.int64)

        hist_cate  = cate_by_adg[hist_adg]
        hist_brand = brand_by_adg[hist_adg]
        hist_price = price_by_adg[hist_adg].astype(np.float32)

        hist_item = np.stack([hist_adg, hist_cate, hist_brand], axis=1).astype(np.int64)
        hist_time = np.stack([hist_year, hist_mon], axis=1).astype(np.int64)
        hist_cont = hist_price.reshape(-1, 1).astype(np.float32)
        lens = hist_item.shape[0]

        neg_cate  = cate_by_adg[neg_adgs]
        neg_brand = brand_by_adg[neg_adgs]
        neg_price = price_by_adg[neg_adgs].astype(np.float32)
        neg_item_cats = np.stack([neg_adgs, neg_cate, neg_brand], axis=1).astype(np.int64)

        return (
            u_feats, hist_item, hist_time, hist_cont,
            pos_item_cats, np.float32(pos_price),
            neg_item_cats, neg_price,
            lens
        )

def collate_fn(batch):
    u_feats, hist_item, hist_time, hist_cont, pos_cats, pos_price, neg_catsK, neg_priceK, lens = zip(*batch)

    lens = torch.tensor(lens, dtype=torch.long)
    order = torch.argsort(lens, descending=True)
    lens = lens[order]

    u_feats = torch.tensor(np.stack(u_feats), dtype=torch.long)[order]
    pos_cats = torch.tensor(np.stack(pos_cats), dtype=torch.long)[order]
    pos_price = torch.tensor(np.array(pos_price, dtype=np.float32), dtype=torch.float32)[order]
    neg_catsK = torch.tensor(np.stack(neg_catsK), dtype=torch.long)[order]
    neg_priceK = torch.tensor(np.stack(neg_priceK).astype(np.float32), dtype=torch.float32)[order]

    B = len(batch)
    Lmax = int(lens.max().item())

    hi_pad = torch.zeros((B, Lmax, 3), dtype=torch.long)
    ht_pad = torch.zeros((B, Lmax, 2), dtype=torch.long)
    hc_pad = torch.zeros((B, Lmax, 1), dtype=torch.float32)

    for bi, src_i in enumerate(order.tolist()):
        L = hist_item[src_i].shape[0]
        hi_pad[bi, :L] = torch.tensor(hist_item[src_i], dtype=torch.long)
        ht_pad[bi, :L] = torch.tensor(hist_time[src_i], dtype=torch.long)
        hc_pad[bi, :L] = torch.tensor(hist_cont[src_i], dtype=torch.float32)

    return u_feats, hi_pad, ht_pad, hc_pad, pos_cats, pos_price, neg_catsK, neg_priceK, lens

train_ds = ClickDatasetActive(user_event, "train", K=K_NEG, max_hist=MAX_HIST_CLICKS)
val_ds   = ClickDatasetActive(user_event, "val",   K=K_NEG, max_hist=MAX_HIST_CLICKS)
test_ds  = ClickDatasetActive(user_event, "test",  K=K_NEG, max_hist=MAX_HIST_CLICKS)

print("Samples (active-only):", {"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)})

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=True, drop_last=True, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=True, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=True, collate_fn=collate_fn
)

# -----------------------
# Model
# -----------------------
class ResidualMLP(nn.Module):
    def __init__(self, din, dh, dout, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(din, dh)
        self.fc2 = nn.Linear(dh, dout)
        self.ln  = nn.LayerNorm(dout)
        self.dp  = nn.Dropout(dropout)
        self.proj = nn.Identity() if din == dout else nn.Linear(din, dout)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = self.dp(h)
        h = self.fc2(h)
        return self.ln(self.proj(x) + h)

class ItemTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.adg_emb   = nn.Embedding(ADG_V,   EMB_DIM, padding_idx=0)
        self.cate_emb  = nn.Embedding(CATE_V,  EMB_DIM, padding_idx=0)
        self.brand_emb = nn.Embedding(BRAND_V, EMB_DIM, padding_idx=0)
        in_dim = EMB_DIM * 3 + 1
        self.m1 = ResidualMLP(in_dim, MLP_HIDDEN, MLP_HIDDEN, DROPOUT)
        self.m2 = ResidualMLP(MLP_HIDDEN, MLP_HIDDEN, EMB_DIM, DROPOUT)

    def forward(self, item_cats_3, price_norm):
        adg, cate, brand = [item_cats_3[:, i] for i in range(3)]
        x = torch.cat([
            self.adg_emb(adg),
            self.cate_emb(cate),
            self.brand_emb(brand),
            price_norm.unsqueeze(-1),
        ], dim=-1)
        x = self.m1(x)
        x = self.m2(x)
        return F.normalize(x, dim=-1)

class HistoryEncoder(nn.Module):
    def __init__(self, mode: str):
        super().__init__()
        self.mode = mode
        self.input_proj = nn.Linear(EMB_DIM * 5 + 1, GRU_HIDDEN)

        if mode == "gru":
            self.gru = nn.GRU(GRU_HIDDEN, GRU_HIDDEN, batch_first=True)

        elif mode == "transformer":
            self.pos_emb = nn.Embedding(MAX_HIST_CLICKS + 2, GRU_HIDDEN, padding_idx=0)
            enc_layer = nn.TransformerEncoderLayer(
                d_model=GRU_HIDDEN,
                nhead=TRANSFORMER_NHEAD,
                dim_feedforward=TRANSFORMER_FF_DIM,
                dropout=TRANSFORMER_DROPOUT,
                activation="gelu",
                batch_first=True,
                norm_first=True
            )
            self.transformer = nn.TransformerEncoder(enc_layer, num_layers=TRANSFORMER_LAYERS)
            self.out_ln = nn.LayerNorm(GRU_HIDDEN)
        else:
            raise ValueError(f"Unknown HISTORY_ENCODER={mode}")

    def forward(self, ev, lens):
        # ev: [B,L,GRU_HIDDEN]
        if self.mode == "gru":
            packed = pack_padded_sequence(ev, lens.cpu(), batch_first=True, enforce_sorted=True)
            _, hN = self.gru(packed)
            return hN[-1]  # [B,H]

        # transformer
        B, L, H = ev.shape
        pos_ids = torch.arange(1, L + 1, device=ev.device).unsqueeze(0).expand(B, L)
        pos_ids = pos_ids.clamp(max=MAX_HIST_CLICKS + 1)
        x = ev + self.pos_emb(pos_ids)

        pad_mask = torch.arange(L, device=ev.device).unsqueeze(0) >= lens.unsqueeze(1)  # [B,L], True=pad
        x = self.transformer(x, src_key_padding_mask=pad_mask)
        x = self.out_ln(x)

        last_idx = (lens - 1).clamp(min=0)
        return x[torch.arange(B, device=ev.device), last_idx]  # [B,H]

class UserTower(nn.Module):
    def __init__(self, shared_item: ItemTower, history_encoder_mode: str = "gru"):
        super().__init__()
        self.user_emb = nn.Embedding(USER_V, EMB_DIM, padding_idx=0)

        self.cms_seg_emb = nn.Embedding(CMS_SEG_V, EMB_DIM, padding_idx=0)
        self.cms_grp_emb = nn.Embedding(CMS_GRP_V, EMB_DIM, padding_idx=0)
        self.gender_emb  = nn.Embedding(GENDER_V,  EMB_DIM, padding_idx=0)
        self.age_emb     = nn.Embedding(AGE_V,     EMB_DIM, padding_idx=0)
        self.shop_emb    = nn.Embedding(SHOP_V,    EMB_DIM, padding_idx=0)
        self.occ_emb     = nn.Embedding(OCC_V,     EMB_DIM, padding_idx=0)

        self.year_ev  = nn.Embedding(YEAR_V,  EMB_DIM, padding_idx=0)
        self.month_ev = nn.Embedding(MONTH_V, EMB_DIM, padding_idx=0)

        self.adg_ev   = shared_item.adg_emb
        self.cate_ev  = shared_item.cate_emb
        self.brand_ev = shared_item.brand_emb

        self.hist_encoder = HistoryEncoder(history_encoder_mode)

        head_in = GRU_HIDDEN + EMB_DIM * (1 + 6)
        self.m1 = ResidualMLP(head_in, MLP_HIDDEN, MLP_HIDDEN, DROPOUT)
        self.m2 = ResidualMLP(MLP_HIDDEN, MLP_HIDDEN, EMB_DIM, DROPOUT)

    def forward(self, u_feats, hi_pad, ht_pad, hc_pad, lens):
        u_code, cms_seg, cms_grp, gender, age, shop, occ = [u_feats[:, i] for i in range(7)]

        e_u = self.user_emb(u_code)
        e_demo = torch.cat([
            self.cms_seg_emb(cms_seg),
            self.cms_grp_emb(cms_grp),
            self.gender_emb(gender),
            self.age_emb(age),
            self.shop_emb(shop),
            self.occ_emb(occ),
        ], dim=-1)

        hadg, hcate, hbrand = [hi_pad[:, :, i] for i in range(3)]
        hyear, hmonth = [ht_pad[:, :, i] for i in range(2)]
        hprice = hc_pad[:, :, 0]

        ev = torch.cat([
            self.adg_ev(hadg),
            self.cate_ev(hcate),
            self.brand_ev(hbrand),
            self.year_ev(hyear),
            self.month_ev(hmonth),
            hprice.unsqueeze(-1),
        ], dim=-1)

        ev = self.hist_encoder.input_proj(ev)
        h_seq = self.hist_encoder(ev, lens)

        x = torch.cat([h_seq, e_u, e_demo], dim=-1)
        x = self.m1(x)
        x = self.m2(x)
        return F.normalize(x, dim=-1)

class TwoTowerSeq(nn.Module):
    def __init__(self, history_encoder_mode: str = "gru"):
        super().__init__()
        self.item = ItemTower()
        self.user = UserTower(self.item, history_encoder_mode=history_encoder_mode)

    def user_vec(self, u_feats, hi_pad, ht_pad, hc_pad, lens):
        return self.user(u_feats, hi_pad, ht_pad, hc_pad, lens)

    def item_vec(self, item_cats_3, price_norm):
        return self.item(item_cats_3, price_norm)

model = TwoTowerSeq(history_encoder_mode=HISTORY_ENCODER).to(DEVICE)
print("History encoder mode:", HISTORY_ENCODER)

# Real explicit L2 => optimizer must NOT secretly add weight decay.
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.0)

# -----------------------
# Temperature schedule
# -----------------------
def tau_schedule(epoch, step, steps_per_epoch):
    if TAU_WARMUP_EPOCHS > 0 and epoch < TAU_WARMUP_EPOCHS:
        t = (epoch * steps_per_epoch + step) / max(TAU_WARMUP_EPOCHS * steps_per_epoch, 1)
        return TAU_START + t * (TAU_END - TAU_START)
    total_epochs = max(EPOCHS - TAU_WARMUP_EPOCHS, 1)
    e2 = epoch - TAU_WARMUP_EPOCHS
    prog = (e2 + step / max(steps_per_epoch, 1)) / total_epochs
    prog = min(max(prog, 0.0), 1.0)
    cos = 0.5 * (1.0 + math.cos(math.pi * prog))
    return TAU_END + (TAU_START - TAU_END) * cos

# -----------------------
# Real explicit L2 regularization
# -----------------------
def explicit_l2_penalty(module, coef, include_embeddings=True, include_bias=True):
    if (coef is None) or (coef <= 0):
        return torch.tensor(0.0, device=DEVICE)

    reg = None
    for name, p in module.named_parameters():
        if not p.requires_grad:
            continue
        if (not include_bias) and name.endswith("bias"):
            continue
        if (not include_embeddings) and ("emb" in name):
            continue

        term = p.pow(2).sum()
        reg = term if reg is None else (reg + term)

    if reg is None:
        return torch.tensor(0.0, device=DEVICE)

    return 0.5 * coef * reg

def total_explicit_l2(model):
    if not USE_EXPLICIT_L2:
        return torch.tensor(0.0, device=DEVICE)

    reg_item = explicit_l2_penalty(
        model.item, L2_ITEM,
        include_embeddings=L2_INCLUDE_EMBEDDINGS,
        include_bias=L2_INCLUDE_BIAS
    )
    reg_user = explicit_l2_penalty(
        model.user, L2_USER,
        include_embeddings=L2_INCLUDE_EMBEDDINGS,
        include_bias=L2_INCLUDE_BIAS
    )
    return reg_item + reg_user

# -----------------------
# Losses: In-batch InfoNCE + sampled NCE (K=255), total = A + 3*B + L2
# -----------------------
def nce_inbatch(u_vec, p_vec, tau):
    logits = (u_vec @ p_vec.t()) / tau
    labels = torch.arange(u_vec.size(0), device=u_vec.device)
    return F.cross_entropy(logits, labels)

def sampled_nce(u_vec, p_vec, n_vecK, tau):
    s_pos = (u_vec * p_vec).sum(dim=-1, keepdim=True)      # [B,1]
    s_neg = (u_vec.unsqueeze(1) * n_vecK).sum(dim=-1)      # [B,K]
    logits = torch.cat([s_pos, s_neg], dim=1) / tau
    labels = torch.zeros((u_vec.size(0),), device=u_vec.device, dtype=torch.long)
    return F.cross_entropy(logits, labels)

@torch.no_grad()
def val_loss_fixed_tau(model, loader, tau=TAU_VAL):
    model.eval()
    tot, n = 0.0, 0
    for batch in loader:
        u_feats, hi_pad, ht_pad, hc_pad, pos_cats, pos_price, neg_catsK, neg_priceK, lens = batch
        u_feats=u_feats.to(DEVICE); hi_pad=hi_pad.to(DEVICE); ht_pad=ht_pad.to(DEVICE); hc_pad=hc_pad.to(DEVICE)
        pos_cats=pos_cats.to(DEVICE); pos_price=pos_price.to(DEVICE)
        neg_catsK=neg_catsK.to(DEVICE); neg_priceK=neg_priceK.to(DEVICE)
        lens=lens.to(DEVICE)

        u = model.user_vec(u_feats, hi_pad, ht_pad, hc_pad, lens)
        p = model.item_vec(pos_cats, pos_price)
        B, K, _ = neg_catsK.shape
        n_flat = model.item_vec(neg_catsK.reshape(B*K, 3), neg_priceK.reshape(B*K)).view(B, K, -1)

        lossA = nce_inbatch(u, p, tau=tau)
        lossB = sampled_nce(u, p, n_flat, tau=tau)
        lossL2 = total_explicit_l2(model)
        loss = lossA + LOSS_B_MULT * lossB + lossL2

        tot += float(loss.item()) * B
        n += B
    return tot / max(n, 1)

# -----------------------
# Active-pool recall@K (within ACTIVE_ADG_CODES only)
# -----------------------
@torch.no_grad()
def build_item_matrix_active(model, chunk=65536):
    model.eval()
    codes = torch.from_numpy(ACTIVE_ADG_CODES).to(device=DEVICE, dtype=torch.long)
    N = int(codes.numel())
    cate = torch.from_numpy(cate_by_adg[ACTIVE_ADG_CODES]).to(device=DEVICE, dtype=torch.long)
    brand = torch.from_numpy(brand_by_adg[ACTIVE_ADG_CODES]).to(device=DEVICE, dtype=torch.long)
    price = torch.from_numpy(price_by_adg[ACTIVE_ADG_CODES]).to(device=DEVICE, dtype=torch.float32)

    mat = torch.empty((N, EMB_DIM), device=DEVICE, dtype=torch.float32)
    for s in range(0, N, chunk):
        e = min(N, s + chunk)
        cats = torch.stack([codes[s:e], cate[s:e], brand[s:e]], dim=1)
        mat[s:e] = model.item_vec(cats, price[s:e])
    return codes, mat

@torch.no_grad()
def recall_at_k_active(model, loader, active_codes, active_mat, Ks=(1,5,10,20,50,100,200,500,1000)):
    model.eval()
    Ks = sorted(Ks)
    maxK = min(Ks[-1], active_mat.size(0))
    Ks = [k for k in Ks if k <= maxK]
    hits = {k: 0 for k in Ks}
    total = 0

    code2row = -np.ones((ADG_V,), dtype=np.int64)
    code2row[active_codes.detach().cpu().numpy()] = np.arange(active_codes.numel(), dtype=np.int64)

    active_T = active_mat.transpose(0, 1).contiguous()

    for batch in loader:
        u_feats, hi_pad, ht_pad, hc_pad, pos_cats, pos_price, neg_catsK, neg_priceK, lens = batch
        u_feats=u_feats.to(DEVICE); hi_pad=hi_pad.to(DEVICE); ht_pad=ht_pad.to(DEVICE); hc_pad=hc_pad.to(DEVICE)
        lens=lens.to(DEVICE)

        uvec = model.user_vec(u_feats, hi_pad, ht_pad, hc_pad, lens)

        pos_adg = pos_cats[:, 0].numpy().astype(np.int64)
        true_row = code2row[pos_adg]
        true_row = torch.from_numpy(true_row).to(device=DEVICE, dtype=torch.long)

        scores = uvec @ active_T
        top_idx = torch.topk(scores, k=maxK, dim=1, largest=True, sorted=False).indices
        hit_mat = (top_idx == true_row.unsqueeze(1))
        for k in Ks:
            hits[k] += int(hit_mat[:, :k].any(dim=1).sum().item())
        total += pos_adg.shape[0]

    return {f"recall@{k}": hits[k] / max(total, 1) for k in Ks}

# -----------------------
# Train
# -----------------------
steps_per_epoch = max(len(train_loader), 1)

for epoch in range(EPOCHS):
    model.train()
    running, seen = 0.0, 0

    for step, batch in enumerate(train_loader, start=0):
        u_feats, hi_pad, ht_pad, hc_pad, pos_cats, pos_price, neg_catsK, neg_priceK, lens = batch
        u_feats=u_feats.to(DEVICE); hi_pad=hi_pad.to(DEVICE); ht_pad=ht_pad.to(DEVICE); hc_pad=hc_pad.to(DEVICE)
        pos_cats=pos_cats.to(DEVICE); pos_price=pos_price.to(DEVICE)
        neg_catsK=neg_catsK.to(DEVICE); neg_priceK=neg_priceK.to(DEVICE)
        lens=lens.to(DEVICE)

        tau = tau_schedule(epoch, step, steps_per_epoch)

        opt.zero_grad(set_to_none=True)

        u = model.user_vec(u_feats, hi_pad, ht_pad, hc_pad, lens)
        p = model.item_vec(pos_cats, pos_price)

        B, K, _ = neg_catsK.shape
        n_flat = model.item_vec(neg_catsK.reshape(B*K, 3), neg_priceK.reshape(B*K)).view(B, K, -1)

        lossA = nce_inbatch(u, p, tau=tau)
        lossB = sampled_nce(u, p, n_flat, tau=tau)
        lossL2 = total_explicit_l2(model)
        loss = lossA + LOSS_B_MULT * lossB + lossL2

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()

        running += float(loss.item()) * B
        seen += B

        if (step + 1) % 50 == 0:
            print(
                f"Epoch {epoch} step {step+1}: tau={tau:.4f} lr={LR:.2e} "
                f"loss={running/seen:.4f} "
                f"(A={lossA.item():.4f}, B={lossB.item():.4f}, L2={lossL2.item():.6f})"
            )

    train_loss = running / max(seen, 1)
    vloss = val_loss_fixed_tau(model, val_loader, tau=TAU_VAL)

    active_codes, active_mat = build_item_matrix_active(model)
    vrec = recall_at_k_active(
        model, val_loader, active_codes, active_mat,
        Ks=(1,5,10,20,50,100,200,500,1000)
    )

    print(
        f"\nEpoch {epoch} done: train_loss={train_loss:.4f}  val_loss={vloss:.4f}  "
        f"val r@5={vrec.get('recall@5', float('nan')):.4f} "
        f"r@10={vrec.get('recall@10', float('nan')):.4f} "
        f"r@20={vrec.get('recall@20', float('nan')):.4f} "
        f"r@50={vrec.get('recall@50', float('nan')):.4f} "
        f"r@100={vrec.get('recall@100', float('nan')):.4f} "
        f"r@200={vrec.get('recall@200', float('nan')):.4f} "
        f"r@500={vrec.get('recall@500', float('nan')):.4f} "
        f"r@1000={vrec.get('recall@1000', float('nan')):.4f}\n"
    )

# -----------------------
# Final test
# -----------------------
active_codes, active_mat = build_item_matrix_active(model)
trec = recall_at_k_active(
    model, test_loader, active_codes, active_mat,
    Ks=(1,5,10,20,50,100,200,500,1000)
)
tloss = val_loss_fixed_tau(model, test_loader, tau=TAU_VAL)

print("TEST (ACTIVE POOL) Recall:", trec)
print("TEST loss (const tau):", tloss)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 10.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.13.0 requires tqdm>=4.67, but you have tqdm 4.66.4 which is incompatible.
dataproc-spark-connect 1.0.2 requires tqdm>=4.67, but you have tqdm 4.66.4 which is incompatible.
DEVICE: cuda
/content/taobao_ad/ 100%[===================>]  29.84M  5.53MB/s    in 5.4s    


/tmp/ipykernel_2427/1060427705.py:54: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(os.path.dirname(path))


/content/taobao_ad/ 100%[===================>]   1.01G  7.91MB/s    in 2m 11s  
/content/taobao_ad/ 100%[===================>]  22.95M  5.18MB/s    in 4.4s    
Files ready
raw rows: 26557961
USER_TOPK chosen=200000 covers click volume=0.7488 (target=0.95)
ADG_TOPK  chosen=171828 covers click volume=0.9500 (target=0.95)
Vocabs: {'USER_V': 200001, 'ADG_V': 171829, 'CATE_V': 6770, 'BRAND_V': 2001, 'YEAR_V': 19}


building user_event: 100%|██████████| 1141729/1141729 [00:52<00:00, 21809.67it/s]


Users kept: 149936
[POOL] min_recent= 50 pool=  3385  val_cov=0.319 (47849/149936)  test_cov=0.317 (47597/149936)
[POOL] min_recent= 30 pool=  6828  val_cov=0.414 (62120/149936)  test_cov=0.412 (61709/149936)
[POOL] min_recent= 18 pool= 13297  val_cov=0.521 (78099/149936)  test_cov=0.518 (77623/149936)
[POOL] min_recent= 10 pool= 26494  val_cov=0.643 (96429/149936)  test_cov=0.640 (95921/149936)
[POOL] min_recent=  6 pool= 46075  val_cov=0.746 (111873/149936)  test_cov=0.741 (111133/149936)
[POOL] min_recent=  3 pool= 91000  val_cov=0.865 (129733/149936)  test_cov=0.862 (129247/149936)
[POOL] min_recent=  1 pool=171828  val_cov=0.951 (142596/149936)  test_cov=0.952 (142690/149936)
Final active pool: size=171828, min_recent=1, val_cov=0.951, test_cov=0.952, end_ts=1494691186, cut_ts=1493481586
Samples (active-only): {'train': 588294, 'val': 142596, 'test': 142690}


/tmp/ipykernel_2427/1060427705.py:609: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=TRANSFORMER_LAYERS)


History encoder mode: transformer
Epoch 0 step 50: tau=0.1978 lr=2.00e-04 loss=23.3644 (A=6.7273, B=5.2823, L2=0.179363)
Epoch 0 step 100: tau=0.1955 lr=2.00e-04 loss=22.9266 (A=6.5880, B=5.1567, L2=0.179087)
Epoch 0 step 150: tau=0.1933 lr=2.00e-04 loss=22.6308 (A=6.4929, B=5.0380, L2=0.178875)
Epoch 0 step 200: tau=0.1910 lr=2.00e-04 loss=22.4085 (A=6.5548, B=5.1059, L2=0.178696)
Epoch 0 step 250: tau=0.1887 lr=2.00e-04 loss=22.2232 (A=6.3458, B=4.8848, L2=0.178531)
Epoch 0 step 300: tau=0.1865 lr=2.00e-04 loss=22.0760 (A=6.4415, B=4.9955, L2=0.178378)
Epoch 0 step 350: tau=0.1842 lr=2.00e-04 loss=21.9533 (A=6.2996, B=4.8434, L2=0.178236)
Epoch 0 step 400: tau=0.1819 lr=2.00e-04 loss=21.8459 (A=6.2600, B=4.8096, L2=0.178102)
Epoch 0 step 450: tau=0.1797 lr=2.00e-04 loss=21.7556 (A=6.2933, B=4.8325, L2=0.177971)
Epoch 0 step 500: tau=0.1774 lr=2.00e-04 loss=21.6726 (A=6.2663, B=4.8148, L2=0.177847)
Epoch 0 step 550: tau=0.1751 lr=2.00e-04 loss=21.5987 (A=6.3003, B=4.8413, L2=0.177732)